In [ ]:
"""
Calculate TOU consumption weights from ResStock building profiles for PGE.

PGE E-TOU-C rate structure:
  Summer (Jun-Oct): Peak 4-9pm, Off-peak all other hours
  Winter (Nov-May): Peak 4-9pm, Off-peak all other hours

Output: tou_weights_pge.csv (4 periods)
"""

import pandas as pd
import numpy as np
from pathlib import Path


def get_tou_period_pge(hour, month):
    """PGE E-TOU-C TOU periods (4 periods, no midpeak)."""
    is_summer = 6 <= month <= 10

    if is_summer:
        if 16 <= hour < 21:
            return 'summer_peak'
        else:
            return 'summer_offpeak'
    else:
        if 16 <= hour < 21:
            return 'winter_peak'
        else:
            return 'winter_offpeak'


def calculate_tou_weights(buildings_dir='./Baseline_PGE'):
    """Calculate TOU weights from all PGE building files in directory."""

    print("=" * 80)
    print("CALCULATING PGE TOU CONSUMPTION WEIGHTS")
    print("=" * 80)

    # Get all parquet files
    building_files = list(Path(buildings_dir).glob('*-0.parquet'))
    print(f"\nFound {len(building_files)} building files")

    # Initialize — 4 PGE TOU periods (no midpeak)
    tou_totals = {
        'summer_peak': 0.0, 'summer_offpeak': 0.0,
        'winter_peak': 0.0, 'winter_offpeak': 0.0,
    }

    # Process each building
    for i, filepath in enumerate(building_files):
        try:
            df = pd.read_parquet(filepath)

            # Aggregate from 15-min to hourly
            load_15min = df['out.electricity.total.energy_consumption'].values
            hourly = load_15min.reshape(-1, 4).sum(axis=1)

            # Classify and sum
            for hour_idx, kwh in enumerate(hourly):
                month = min((hour_idx // 730) + 1, 12)
                hour = hour_idx % 24
                period = get_tou_period_pge(hour, month)
                tou_totals[period] += kwh

            if (i + 1) % 2000 == 0:
                print(f"  Processed {i+1}/{len(building_files)} buildings")

        except Exception as e:
            print(f"  Error with {filepath.name}: {e}")
            continue

    # Calculate weights
    total = sum(tou_totals.values())
    tou_weights = {k: v / total for k, v in tou_totals.items()}

    # Display
    print("\n" + "=" * 80)
    print("PGE TOU CONSUMPTION WEIGHTS (E-TOU-C)")
    print("=" * 80)
    print(f"\nTotal consumption: {total:,.0f} kWh")
    for period, weight in tou_weights.items():
        print(f"  {period:20s}: {weight:.4f} ({weight*100:.2f}%)")

    return tou_weights


# Run and save
tou_weights = calculate_tou_weights()
df = pd.DataFrame(list(tou_weights.items()), columns=['period', 'weight'])
df.to_csv('tou_weights_pge.csv', index=False)
print("\n" + "=" * 80)
print("Saved to: tou_weights_pge.csv")
print("=" * 80)